# Attention Pattern Motifs

Detect common attention pattern structures: diagonal, stripe, block,
triangular, and motif classification summary.

## Why This Matters

Attention pattern motifs reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_motifs import (
    detect_diagonal_motif, detect_stripe_motif,
    detect_block_motif, detect_triangular_motif,
    attention_motif_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Diagonal Motif

Previous-token and self-attention patterns.

In [ ]:
for layer in range(4):
    for head in range(4):
        r = detect_diagonal_motif(model, tokens, layer, head)
        flags = []
        if r['is_diagonal']: flags.append('DIAG')
        if r['is_self_attention']: flags.append('SELF')
        if flags:
            print(f"  L{layer}H{head}: diag={r['diagonal_score']:.3f}, "
                  f"self={r['self_attention_score']:.3f} [{', '.join(flags)}]")

## Stripe Motif

Column-dominant patterns (one key position gets all attention).

In [ ]:
for layer in range(4):
    for head in range(4):
        r = detect_stripe_motif(model, tokens, layer, head)
        flag = ' [STRIPE]' if r['is_stripe'] else ''
        print(f"  L{layer}H{head}: stripe={r['stripe_score']:.3f}, "
              f"col={r['dominant_column']}, conc={r['column_concentration']:.3f}{flag}")

## Block/Local Motif

Attention concentrated on nearby positions.

In [ ]:
for layer in range(4):
    for head in range(4):
        r = detect_block_motif(model, tokens, layer, head)
        flag = ' [LOCAL]' if r['is_local'] else ''
        print(f"  L{layer}H{head}: local={r['local_score']:.3f}, "
              f"window={r['window_size']}{flag}")

## Motif Summary

Classify all heads by dominant attention motif.

In [ ]:
result = attention_motif_summary(model, tokens)
print(f"Most common motif: {result['most_common_motif']}")
print(f"Motif counts: {result['motif_counts']}\n")
for c in result['classifications']:
    scores = ', '.join(f'{k}:{v:.2f}' for k, v in c['scores'].items())
    print(f"  L{c['layer']}H{c['head']}: {c['dominant_motif']:12s} | {scores}")